In [2]:
import json

# Load the JSON file
def load_data(filename):
    with open(filename, "r") as file:
        data = json.load(file)
    return data

# Display users and their connections
def display_users(data):
    print("Users and Their Connections:\n")
    for user in data["users"]:
        print(f"{user['name']} (ID: {user['id']}) - Friends: {user['friends']} - Liked Pages: {user['liked_pages']}")
    print("\nPages:\n")
    for page in data["pages"]:
        print(f"{page['id']}: {page['name']}")

# Load and display the data
data = load_data("codebook_data.json")
display_users(data)

Users and Their Connections:

Amit (ID: 1) - Friends: [2, 3, 4] - Liked Pages: [101, 102, 103]
Priya (ID: 2) - Friends: [1, 5, 6] - Liked Pages: [101, 104]
Rahul (ID: 3) - Friends: [1, 7] - Liked Pages: [102, 103]
Sara (ID: 4) - Friends: [1, 8] - Liked Pages: [105]
Kiran (ID: 5) - Friends: [2, 9] - Liked Pages: [101, 106]
Anjali (ID: 6) - Friends: [2, 10] - Liked Pages: [103, 104]
Vikram (ID: 7) - Friends: [3, 11] - Liked Pages: [102, 107]
Neha (ID: 8) - Friends: [4, 12] - Liked Pages: [101, 105]
Ravi (ID: 9) - Friends: [5, 13] - Liked Pages: [106, 108]
Pooja (ID: 10) - Friends: [6, 14] - Liked Pages: [101, 102]
Arjun (ID: 11) - Friends: [7, 15] - Liked Pages: [107, 108]
Meera (ID: 12) - Friends: [8] - Liked Pages: [101, 104]
Sanjay (ID: 13) - Friends: [9] - Liked Pages: [106, 102]
Divya (ID: 14) - Friends: [10] - Liked Pages: [103, 105]
Rohit (ID: 15) - Friends: [11] - Liked Pages: [101, 107]
Amit (ID: 16) - Friends: [2, 3, 4] - Liked Pages: [101, 102, 103]
Priya (ID: 17) - Friends: [

In [3]:
import json

def clean_data(data):
    # Remove users with missing names
    data["users"] = [user for user in data["users"] if user["name"].strip()]
    
    # Remove duplicate friends
    for user in data["users"]:
        user["friends"] = list(set(user["friends"]))
    
    # Remove inactive users
    data["users"] = [user for user in data["users"] if user["friends"] or user["liked_pages"]]
    
    # Remove duplicate pages
    unique_pages = {}
    for page in data["pages"]:
        unique_pages[page["id"]] = page
    data["pages"] = list(unique_pages.values())
    
    return data

# Load, clean, and display the cleaned data
data = json.load(open("codebook_data.json"))
data = clean_data(data)
json.dump(data, open("cleaned_codebook_data.json", "w"), indent=4)
print("Data cleaned successfully!")

Data cleaned successfully!


In [7]:
def find_people_you_may_know(user_id, data):
    user_friends = {}
    
    for user in data["users"]:
        # Handle null or missing friends safely
        friends = user.get("friends") or []
        user_friends[user["id"]] = set(friends)
    
    if user_id not in user_friends:
        return []
    
    direct_friends = user_friends[user_id]
    suggestions = {}
    
    for friend in direct_friends:
        if friend not in user_friends:
            continue  # skip invalid users
        
        for mutual in user_friends[friend]:
            if mutual != user_id and mutual not in direct_friends:
                suggestions[mutual] = suggestions.get(mutual, 0) + 1
    
    sorted_suggestions = sorted(suggestions.items(), key=lambda x: x[1], reverse=True)
    return [uid for uid, _ in sorted_suggestions]


# Load data
data = load_data("cleaned_codebook_data.json")

# Take input from user
user_id = int(input("Enter User ID to find 'People You May Know': "))

# Get recommendations
recommendations = find_people_you_may_know(user_id, data)

# Output
print(f"\nPeople You May Know for User {user_id}: {recommendations}")

Enter User ID to find 'People You May Know':  4



People You May Know for User 4: [12, 2, 3]


In [9]:
import json

# Function to find pages a user might like
def find_pages_you_might_like(user_id, data):
    user_pages = {}
    
    for user in data["users"]:
        # Handle null safely
        pages = user.get("liked_pages") or []
        user_pages[user["id"]] = set([p for p in pages if p is not None])
    
    if user_id not in user_pages:
        return []
    
    user_liked_pages = user_pages[user_id]
    page_suggestions = {}
    
    for other_user, pages in user_pages.items():
        if other_user != user_id:
            shared_pages = user_liked_pages.intersection(pages)
            
            for page in pages:
                if page not in user_liked_pages:
                    page_suggestions[page] = page_suggestions.get(page, 0) + len(shared_pages)
    
    sorted_pages = sorted(page_suggestions.items(), key=lambda x: x[1], reverse=True)
    return [page_id for page_id, _ in sorted_pages]


# Load data
data = load_data("cleaned_codebook_data.json")

# Take input from user
user_id = int(input("Enter User ID to find 'Pages You Might Like': "))

# Get recommendations
page_recommendations = find_pages_you_might_like(user_id, data)

# Output
print(f"\nPages You Might Like for User {user_id}: {page_recommendations}")

Enter User ID to find 'Pages You Might Like':  8



Pages You Might Like for User 8: [104, 102, 103, 106, 107, 108, 110]
